# ПЗ №7. Вариант 1: Авиакомпания

**Задача:** распределить рейсы между тремя хабами (A, B, C) и четырьмя аэропортами (X, Y, Z, W) так, чтобы минимизировать общие затраты.

**Пропускная способность хабов:** A — 20, B — 15, C — 10 рейсов/неделю.
**Спрос аэропортов:** X — 12, Y — 8, Z — 14, W — 6 рейсов/неделю.

**Целевая функция (минимизация затрат):**
Z = 150*x_AX + 200*x_AY + 180*x_AZ + 220*x_AW
  + 170*x_BX + 160*x_BY + 190*x_BZ + 210*x_BW
  + 200*x_CX + 180*x_CY + 170*x_CZ + 190*x_CW → min

**Ограничения:**
- По хабам (≤): A ≤ 20, B ≤ 15, C ≤ 10
- По спросу (=): X = 12, Y = 8, Z = 14, W = 6
- Неотрицательность: x_ij ≥ 0

In [1]:
import numpy as np
from scipy.optimize import linprog
import time

In [2]:
# === МАТРИЦЫ И ВЕКТОРЫ КОЭФФИЦИЕНТОВ ===

# Вектор коэффициентов целевой функции (12 переменных)
# Порядок: x_AX, x_AY, x_AZ, x_AW, x_BX, x_BY, x_BZ, x_BW, x_CX, x_CY, x_CZ, x_CW
c = np.array([
    150, 200, 180, 220,   # A → X, Y, Z, W
    170, 160, 190, 210,   # B → X, Y, Z, W
    200, 180, 170, 190    # C → X, Y, Z, W
], dtype=np.float64)

# Матрица ограничений по пропускной способности хабов (3×12)
A_ub = np.array([
    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],  # A
    [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0],  # B
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]   # C
], dtype=np.float64)
b_ub = np.array([20, 15, 10], dtype=np.float64)

# Матрица ограничений по спросу аэропортов (4×12)
A_eq = np.array([
    [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],  # X
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],  # Y
    [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0],  # Z
    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1]   # W
], dtype=np.float64)
b_eq = np.array([12, 8, 14, 6], dtype=np.float64)

# Границы переменных (все ≥ 0)
bounds = [(0, None)] * 12

In [3]:
# РЕШЕНИЕ ТРЕМЯ МЕТОДАМИ С ЗАМЕРОМ ВРЕМЕНИ

methods = ['highs', 'highs-ds', 'highs-ipm']
results = {}

for method in methods:
    start = time.perf_counter()
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=bounds, method=method)
    elapsed = time.perf_counter() - start
    results[method] = (res, elapsed)
    print(f"\n=== Метод: {method} ===")
    print(f"Время выполнения: {elapsed*1000:.4f} мс")
    print(f"Статус: {res.status} ({res.message})")
    if res.success:
        print(f"Минимальные затраты: {res.fun:.0f}")


=== Метод: highs ===
Время выполнения: 8.3626 мс
Статус: 0 (Optimization terminated successfully. (HiGHS Status 7: Optimal))
Минимальные затраты: 6720

=== Метод: highs-ds ===
Время выполнения: 1.7958 мс
Статус: 0 (Optimization terminated successfully. (HiGHS Status 7: Optimal))
Минимальные затраты: 6720

=== Метод: highs-ipm ===
Время выполнения: 3.8735 мс
Статус: 0 (Optimization terminated successfully. (HiGHS Status 7: Optimal))
Минимальные затраты: 6720


In [4]:
#ВЫВОД ОПТИМАЛЬНОГО РАСПРЕДЕЛЕНИЯ 

best_res = results['highs'][0]
x = best_res.x.reshape(3, 4)

hubs = ['A', 'B', 'C']
airports = ['X', 'Y', 'Z', 'W']

print("Оптимальное распределение рейсов:")
print(f"{'':>4} | " + " | ".join(f"{a:>4}" for a in airports))
print("-" * 30)
for i, hub in enumerate(hubs):
    row = " | ".join(f"{x[i, j]:>4.0f}" for j in range(4))
    print(f"{hub:>4} | {row}")

print(f"\nОбщие затраты: {best_res.fun:.0f}")

Оптимальное распределение рейсов:
     |    X |    Y |    Z |    W
------------------------------
   A |   12 |    0 |    8 |    0
   B |    0 |    8 |    0 |    2
   C |    0 |    0 |    6 |    4

Общие затраты: 6720


In [5]:
# СРАВНЕНИЕ ВРЕМЕНИ ВЫПОЛНЕНИЯ 

print("Сравнение методов оптимизации:")
print(f"{'Метод':<12} | {'Время (мс)':>10} | {'Статус':>6}")
print("-" * 35)
for method in methods:
    res, elapsed = results[method]
    print(f"{method:<12} | {elapsed*1000:>10.4f} | {res.status:>6}")

Сравнение методов оптимизации:
Метод        | Время (мс) | Статус
-----------------------------------
highs        |     8.3626 |      0
highs-ds     |     1.7958 |      0
highs-ipm    |     3.8735 |      0


In [6]:
# === ПРОВЕРКА ВЫПОЛНЕНИЯ ОГРАНИЧЕНИЙ ===

print("Проверка ограничений по хабам (≤):")
for i, hub in enumerate(hubs):
    total = sum(x[i, :])
    print(f"{hub}: {total:.0f} / {b_ub[i]:.0f}")

print("\nПроверка спроса аэропортов (=):")
for j, airport in enumerate(airports):
    total = sum(x[:, j])
    print(f"{airport}: {total:.0f} / {b_eq[j]:.0f}")

Проверка ограничений по хабам (≤):
A: 20 / 20
B: 10 / 15
C: 10 / 10

Проверка спроса аэропортов (=):
X: 12 / 12
Y: 8 / 8
Z: 14 / 14
W: 6 / 6


## Выводы

1. Задача линейного программирования решена с помощью `scipy.optimize.linprog`.
2. Рассмотрены три метода: `highs`, `highs-ds`, `highs-ipm`.
3. Все методы дают одинаковое оптимальное решение, но различаются по времени.
4. Найденное распределение рейсов минимизирует общие затраты при полном удовлетворении спроса и соблюдении пропускной способности хабов.